In [ ]:
import warnings
warnings.simplefilter(action='ignore')

import argparse
import os,cv2
import random,numpy,pandas
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Must come before torch


In [ ]:
import argparse
import os
import random,numpy
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.optim as optim
import torch.nn.functional as F
import torch.utils.data
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import torchvision.utils as vutils
from torchvision.models.feature_extraction import create_feature_extractor
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True
torch.use_deterministic_algorithms(False)


In [ ]:
ngpu=1
ngf,nc = 3,3
ndf = 64

fsc_test={}

transform = transforms.Compose([
    transforms.Resize((450,450)),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
z=['WhatsApp Image 2025-01-08 at 4.06.35 PM.jpeg','WhatsApp Image 2025-01-23 at 3.53.52 PM.jpeg']
for i in z:
    cv2.imwrite(f'/kaggle/working/{i}' ,cv2.cvtColor(cv2.imread(f'/kaggle/input/total-mata/{i}'),cv2.COLOR_BGR2HSV))
    fsc_test[i]=transform(Image.open(f'/kaggle/working/{i}'))

device = torch.device("cuda:0" if (torch.cuda.is_available() and ngpu > 0) else "cpu")

In [ ]:
plt.imshow(cv2.cvtColor(cv2.imread(f'/kaggle/input/total-mata/WhatsApp Image 2025-01-08 at 4.06.35 PM.jpeg'), 
                       cv2.COLOR_BGR2RGB))

In [ ]:
plt.imshow(cv2.imread(f'/kaggle/working/{z[0]}'))

In [ ]:
plt.imshow(cv2.cvtColor(cv2.imread(f'/kaggle/input/total-mata/WhatsApp Image 2025-01-23 at 3.53.52 PM.jpeg'), 
                       cv2.COLOR_BGR2RGB))

In [ ]:
plt.imshow(cv2.imread(f'/kaggle/working/{z[1]}'))

In [ ]:
class EffnetModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(weights=torchvision.models.efficientnet.EfficientNet_V2_M_Weights.DEFAULT)
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = torch.nn.Sequential(
            torch.nn.Linear(1280, 1)
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

In [ ]:
EFF_NET = EffnetModel().float()
EFF_NET= nn.DataParallel(EFF_NET).to(device)
#EFF_NET
#EFF_NET.load_state_dict(torch.load("/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/0.00015554654237348586 87.0.pth",weights_only=False,map_location=torch.device('cpu')))

In [ ]:
fsc_submission=pandas.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col ="image")

In [ ]:
sub = [0]*len(z)
file_ = ['0.00010798667062772438 47.pth']#os.listdir('/kaggle/input/d/mafiosoquasar/fake-scene-classification-model-2/')

for j in file_:
    print(j)
    z_add,z_total=0,0
    EFF_NET.load_state_dict(torch.load(f"/kaggle/input/fake-scene-classification-model-2/{j}", 
                                       weights_only=False, map_location=torch.device('cpu')))
    for i,j in zip(z,range(len(z))):
        
        img=fsc_test[i].reshape((1, 3, 450, 450)).float().to(device)
        sub[j] += EFF_NET(img).sigmoid().cpu().detach().numpy()[0][0]

In [ ]:
submission=pandas.DataFrame({'image' : z, 
                             'label' : 1-numpy.array(sub)})
pandas.DataFrame(submission).to_csv(f"submission.csv", index=False)
pandas.DataFrame(submission)

In [ ]:
import warnings
warnings.simplefilter(action='ignore')

import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision.models.feature_extraction import create_feature_extractor
from torch.utils.data import DataLoader, TensorDataset
from PIL import Image
import matplotlib.pyplot as plt

# Set deterministic behavior
seed = 999
print("Random Seed: ", seed)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(False)

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Show an example image
plt.imshow(cv2.cvtColor(cv2.imread('/kaggle/input/total-mata/WhatsApp Image 2025-01-08 at 4.06.35 PM.jpeg'), cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

# Hyperparameters
batch_size = 8
learning_rate = 0.00001  # Used if training

# Model class
class EffnetModel(nn.Module):
    def __init__(self):
        super().__init__()
        effnet = torchvision.models.efficientnet_v2_m(
            weights=torchvision.models.EfficientNet_V2_M_Weights.DEFAULT
        )
        self.model = create_feature_extractor(effnet, ['flatten'])
        self.nn_fracture = nn.Sequential(
            nn.Linear(1280, 1)
        )
    def forward(self, x):
        x = self.model(x)['flatten']
        x = self.nn_fracture(x)
        return x

# Initialize model
EFF_NET = EffnetModel().float()
EFF_NET = nn.DataParallel(EFF_NET).to(device)

# Load model weights
model_path = "/kaggle/input/fake-scene-classification-model-2/0.00010798667062772438 47.pth"
EFF_NET.load_state_dict(torch.load(model_path, map_location=device))

# Load sample submission to get image names
submission_df = pd.read_csv("/kaggle/input/cidaut-ai-fake-scene-classification-2024/sample_submission.csv", index_col="image")
z = submission_df.index.tolist()

# Load and preprocess test images
def preprocess_image(img_path):
    img = Image.open(img_path).convert("RGB")
    transform = torchvision.transforms.Compose([
        torchvision.transforms.Resize((450, 450)),
        torchvision.transforms.ToTensor(),
    ])
    return transform(img)

fsc_test = []
test_image_dir = "/kaggle/input/cidaut-ai-fake-scene-classification-2024/Test"
for image_name in z:
    img_path = os.path.join(test_image_dir, image_name)
    img_tensor = preprocess_image(img_path)
    fsc_test.append(img_tensor)

fsc_test_tensor = torch.stack(fsc_test)  # Shape: (N, 3, 450, 450)

# Create DataLoader for inference
test_dataset = TensorDataset(fsc_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Run inference
EFF_NET.eval()
sub = []

with torch.no_grad():
    for batch in test_loader:
        imgs = batch[0].to(device)
        preds = EFF_NET(imgs).sigmoid().cpu().numpy().flatten()
        sub.extend(preds)

# Save submission
submission = pd.DataFrame({
    'image': z,
    'label': 1 - np.array(sub)
})
submission.to_csv("submission.csv", index=False)
print(submission.head())


In [ ]:
from PIL import Image
import torch
import torchvision.transforms as transforms
import torchvision

# Function to predict a single image
def predict_image(image_path, model, device):
    model.eval()
    
    # Define transform
    transform = transforms.Compose([
        transforms.Resize((450, 450)),
        transforms.ToTensor(),
    ])
    
    # Load and preprocess image
    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)  # Shape: (1, 3, 450, 450)

    # Forward pass
    with torch.no_grad():
        output = model(img_tensor)
        probability = torch.sigmoid(output).item()
    
    print(f"Predicted probability of being FAKE: {probability:.4f}")
    print(f"Label: {'FAKE' if probability < 0.6 else 'REAL'}")
    return probability

# Example usage
image_path = ['/kaggle/input/govindan-mata/edit.jpg','/kaggle/input/govindan-mata/edit2.jpg','/kaggle/input/govindan-mata/edit1.jpg','/kaggle/input/govindan-mata/edit3.jpg','/kaggle/input/govindan-mata/edit4.jpg','/kaggle/input/govindan-mata/edit5.jpg']
for i in image_path:
    predict_image(i, EFF_NET, device)
